# 03 – So sánh M0–M4, ablation M3, phân tích lỗi

Với mỗi ứng viên, mỗi phương pháp xếp ~10 tin; so thứ tự với nhãn tay. M4 dùng điểm trên **fold kiểm tra** do notebook 02 lưu (`m4_oof.csv`). Kết quả ở đây là kết luận duy nhất về hiệu quả mô hình; dữ liệu ứng tuyển mô phỏng không dùng để kết luận.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

# Có thể đổi bằng biến môi trường khi chạy thử trên dữ liệu khác
DATA = Path(os.getenv("TH3_DATA_DIR", ROOT / "backend/seed/data"))
LABELS = Path(os.getenv("TH3_LABELS_DIR", ROOT / "backend/seed/labels"))
OUTDIR = Path(os.getenv("TH3_OUT_DIR", ROOT / "docs/report/tables"))
MODELS_OUT = Path(os.getenv("TH3_MODELS_OUT", ROOT / "backend/models_store"))
OUTDIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd
from app.ai import datasets, metrics
from app.ai.embedder import get_embedder

emb = get_embedder()
print("Embedder:", emb.name)  # st:... là model thật; hash-384 chỉ để chạy thử, KHÔNG dùng số liệu để báo cáo
jobs = datasets.load_jobs(DATA / "jobs.csv")
cands = datasets.load_candidates(DATA / "candidates_generated.csv")
datasets.attach_embeddings(jobs, emb)
datasets.attach_embeddings(cands, emb)

LABEL_A = os.getenv("TH3_LABELS_A", "labels_pairs_hung.csv")
LABEL_B = os.getenv("TH3_LABELS_B", "labels_pairs_hoanganh.csv")
raw = datasets.load_labels(LABELS / LABEL_A, LABELS / LABEL_B)
col_a, col_b = [c for c in raw.columns if c.startswith("label_")]
print(f"{len(raw)} cặp có đủ nhãn của cả hai người")


In [ ]:
kappa = metrics.weighted_kappa(raw[col_a].to_numpy(), raw[col_b].to_numpy())
print(f"Cohen's kappa (trọng số bình phương) = {kappa:.3f}  (mục tiêu >= 0,6)")

raw["label"] = metrics.merge_labels(raw[col_a].to_numpy(), raw[col_b].to_numpy())
disputed = raw[raw["label"] < 0]
print(f"{len(disputed)} cặp lệch >= 2 bậc cần hai người thảo luận và chốt lại")
disputed.to_csv(OUTDIR / "labels_disputed.csv", index=False)
labels = raw[raw["label"] >= 0].reset_index(drop=True)  # SAU KHI thảo luận: cập nhật file nhãn rồi chạy lại ô này
labels[["candidate_id", "job_id", "label"]].to_csv(LABELS / "labels_pairs_final.csv", index=False)
pd.DataFrame([{"kappa_quadratic": kappa, "pairs": len(raw), "disputed": len(disputed)}]).to_csv(OUTDIR / "kappa.csv", index=False)


In [ ]:
from app.ai import evaluate

oof = pd.read_csv(OUTDIR / "m4_oof.csv", dtype={"candidate_id": str, "job_id": str})
m4 = labels.merge(oof, on=["candidate_id", "job_id"], how="left")["m4_oof_score"].to_numpy()
assert not np.isnan(m4).any(), "m4_oof.csv không khớp nhãn hiện tại: chạy lại notebook 02"
comparison = evaluate.evaluate_models(labels, cands, jobs, m4_scores=m4, embedder=emb)
comparison.to_csv(OUTDIR / "model_comparison.csv")
comparison.round(4)

In [ ]:
_, feats = evaluate.build_feature_matrix(labels, cands, jobs, emb)
ablation = evaluate.ablation_m3(labels, feats)
ablation.to_csv(OUTDIR / "ablation_m3.csv", index=False)
ablation.round(4)

In [ ]:
worst = evaluate.error_analysis(labels, cands, jobs, model="m3", n=5, embedder=emb)
worst.to_csv(OUTDIR / "error_analysis_m3.csv", index=False)
worst

## Công bằng

Đổi tên/giới tính ở phần đầu CV, kết quả phải không đổi. Cần hai file CV chỉ khác phần đầu: đặt đường dẫn vào `FAIR_A`, `FAIR_B` (tuỳ chọn).

In [ ]:
FAIR_A, FAIR_B = os.getenv("TH3_FAIR_A"), os.getenv("TH3_FAIR_B")
if FAIR_A and FAIR_B:
    job = next(iter(jobs.values()))
    res = evaluate.fairness_check(Path(FAIR_A).read_bytes(), Path(FAIR_B).read_bytes(), Path(FAIR_A).name, job, embedder=emb)
    print(res)
else:
    print("Bỏ qua: chưa cung cấp cặp CV để kiểm tra công bằng.")